# **Hybrid RAG**

Hybrid RAG refers to an advanced retrieval technique that combines vector similarity search with traditional search methods, such as full-text search or BM25. This approach enables more comprehensive and flexible information retrieval by leveraging the strengths of both methods, vector similarity for semantic understanding and traditional techniques for precise keyword or text-based matching.

Research Paper: [paper1](https://arxiv.org/pdf/2408.05141) and [paper2](https://arxiv.org/pdf/2408.04948)

## **Initial Setup**

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# load embedding model
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 모델 변경
    # dimensions=512,                  # 임베딩 차원 축소 (3 계열에서만 지원)
    # chunk_size=500,                  # 한 번에 임베딩 요청할 텍스트 개수
    # max_retries=3,                   # 실패 시 재시도 횟수
)

In [3]:
# load data
from langchain.document_loaders import CSVLoader
loader = CSVLoader("../data/context.csv", encoding="utf-8")
documents = loader.load()

In [4]:
print(documents[0])
print("-"*100)
print(len(documents))

page_content='context: ['African immigration to the United States refers to immigrants to the United States who are or were nationals of modern African countries. The term African in the scope of this article refers to geographical or national origins rather than racial affiliation. Between the Immigration and Nationality Act of 1965 and 2017, Sub-Saharan African-born population in the United States grew to 2.1 million people.Sub-Saharan Africans in the United States come from almost all regions in Africa and do not constitute a homogeneous group. They include peoples from different national, linguistic, ethnic, racial, cultural and social backgrounds. As such, US and foreign born Sub-Saharan Africans are distinct from native-born African Americans, many of whose ancestors were involuntarily brought from West and Central Africa to the colonial United States by means of the historic Atlantic slave trade. African immigration is now driving the growth of the Black population in New York C

In [5]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

In [6]:
len(documents)

3693

In [12]:
# create vectorstore
from langchain_chroma import Chroma
import chromadb
chromadb.api.client.SharedSystemClient.clear_system_cache()
os.environ["ANONYMIZED_TELEMETRY"] = "False"

vectorstore = Chroma.from_documents(
    documents,
    embeddings,
    persist_directory="../database/chroma_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
print(vectorstore._collection.count()) # 저장된 벡터 개수가 찍히면 정상

3693


## **Retrievers**

In [18]:
# create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",        # 검색 방식
    search_kwargs={"k": 4}           # 검색 세부 옵션 (딕셔너리)
)

### **Keyword Retriever**

BM25 방식은 키워드 서치 방식으로 문서, 질문에서 키워드를 보고 찾는 방식이라 임베딩을 하지 않고 질문이 들어오면 토큰화를 진행한 다음에 TF-IDF 방식으로 질문의 단어가 문서에 자주 출범하면서도 흔한 단어는 제외하고 드문 단어에 가중치를 높여서 핵심 키워드 위주로 매칭하는 방식임

In [19]:
# create keyword retriever
from langchain.retrievers import BM25Retriever
keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k =  3

In [ ]:
# test keyword retriever
keyword_retriever.invoke("what bacteria grow on macconkey agar")

[Document(metadata={'source': '../data/context.csv', 'row': 6}, page_content='predominantly made from the lactose sugar in the agar.\\n\\n\\n== Variant ==\\nA variant, sorbitol-MacConkey agar, (with the addition of additional selective agents) can assist in the isolation and differentiation of enterohemorrhagic E. coli serotype E. coli O157:H7, by the presence of colorless circular colonies that are non-sorbitol fermenting.\\n\\n\\n== See also ==\\nR2a agar\\nMRS agar (culture medium designed to grow Gram-positive bacteria and differentiate them for lactose fermentation).\\n\\n\\n=='),
 Document(metadata={'source': '../data/context.csv', 'row': 37}, page_content='zoonotic disease since around 1910, but in the 1930s knowledge was gained that the bacteria lost their virulent power when repeatedly spread on agar media. This explained the difficulties to reproduce results from different studies as the pre-inoculating handlings of the bacteria were not standardized among scientists.Today it

### **Ensemble Retriever**

기존 chroma db의 임베딩이 비슷한 공간에 있으면 유사도가 높은 방식의 semantic search와 방금 bm25의 키워드 서치 방식을 같이 사용하는 하이브리드 서치

In [22]:
# create ensemble retriever
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(retrievers=[retriever, keyword_retriever], weights=[0.5, 0.5])

In [23]:
# test ensemble retriever
ensemble_retriever.invoke("what bacteria grow on macconkey agar")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='adc763ad-c685-4cbb-8e2d-47862a92a0d7', metadata={'row': 6, 'source': '../data/context.csv'}, page_content="context: ['MacConkey agar is a selective and differential culture medium for bacteria. It is designed to selectively isolate Gram-negative and enteric (normally found in the intestinal tract) bacteria and differentiate them based on lactose fermentation. Lactose fermenters turn red or pink on MacConkey agar, and nonfermenters do not change color. The media inhibits growth of Gram-positive organisms with crystal violet and bile salts, allowing for the selection and isolation of gram-negative"),
 Document(id='c96693c0-a4d0-4dec-bddb-adb599c8b527', metadata={'row': 6, 'source': '../data/context.csv'}, page_content='predominantly made from the lactose sugar in the agar.\\n\\n\\n== Variant ==\\nA variant, sorbitol-MacConkey agar, (with the addition of additional selective agents) can assist in the isolation and differentiation of enterohemorrhagic E. coli serotype E. coli

## **RAG Chain**

In [ ]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

In [52]:
# create document chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

template = """"
You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.
Context: {context}

Question: {input}

Answer:

"""
prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": ensemble_retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [53]:
# response
response = rag_chain.invoke('what bacteria grow on macconkey agar')
response

'MacConkey agar is designed to selectively isolate Gram-negative and enteric bacteria. It differentiates these bacteria based on lactose fermentation. Gram-negative bacteria that are lactose fermenters will turn red or pink on MacConkey agar, while nonfermenters do not change color. The media inhibits the growth of Gram-positive organisms with crystal violet and bile salts, allowing for the selection and isolation of Gram-negative bacteria.'

## **Preparing Data for Evaluation**

In [54]:
# create dataset
questions = ["what bacteria grow on macconkey agar", "who wrote a rose is a rose is a rose"]
response = []
contexts = []

# Inference
for query in questions:
  response.append(rag_chain.invoke(query))
  contexts.append([docs.page_content for docs in ensemble_retriever.invoke(query)])

# To dict
data = {
    "query": questions,
    "response": response,
    "context": contexts,
}

In [55]:
# create dataset
from datasets import Dataset
dataset = Dataset.from_dict(data)

In [56]:
# create dataframe
import pandas as pd
df = pd.DataFrame(dataset)

In [57]:
df

,query,response,context
0,what bacteria grow on macconkey agar,MacConkey agar is designed to selectively isol...,[context: ['MacConkey agar is a selective and ...
1,who wrote a rose is a rose is a rose,"Gertrude Stein wrote the sentence ""A rose is a...",[she would carve on the tree Rose is a Rose is...


In [58]:
df['context'][1][1]

'context: [\'The sentence "Rose is a rose is a rose is a rose" was written by Gertrude Stein as part of the 1913 poem "Sacred Emily", which appeared in the 1922 book Geography and Plays. In that poem, the first "Rose" is the name of a person. Stein later used variations on the sentence in other writings, and the shortened form "A rose is a rose is a rose" is among her most famous quotations, often interpreted as meaning "things are what they are", a statement of the law of identity, "A is A". \\nIn'

In [59]:
# Convert to dictionary
df_dict = df.to_dict(orient='records')

# Convert context to list
for record in df_dict:
    if not isinstance(record.get('context'), list):
        if record.get('context') is None:
            record['context'] = []
        else:
            record['context'] = [record['context']]

## **Evaluation with Ragas**

[Ragas](https://docs.ragas.io/)로 RAG 파이프라인을 평가합니다.

- **Faithfulness**: 답변이 검색된 컨텍스트에 근거하는지 (환각 탐지)
- **Answer Relevancy**: 답변이 질문에 얼마나 관련 있는지

In [60]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy

# ragas 0.2.x: from_dict()는 list of dicts 형태 필요
eval_data = [
    {
        "user_input": q,
        "response": r,
        "retrieved_contexts": c,
    }
    for q, r, c in zip(data["query"], data["response"], data["context"])
]
ragas_dataset = EvaluationDataset.from_dict(eval_data)

In [61]:
eval_data

[{'user_input': 'what bacteria grow on macconkey agar',
  'response': 'MacConkey agar is designed to selectively isolate Gram-negative and enteric (normally found in the intestinal tract) bacteria. It differentiates them based on lactose fermentation. The medium inhibits the growth of Gram-positive organisms, allowing for the selection and isolation of Gram-negative bacteria.',
  'retrieved_contexts': ["context: ['MacConkey agar is a selective and differential culture medium for bacteria. It is designed to selectively isolate Gram-negative and enteric (normally found in the intestinal tract) bacteria and differentiate them based on lactose fermentation. Lactose fermenters turn red or pink on MacConkey agar, and nonfermenters do not change color. The media inhibits growth of Gram-positive organisms with crystal violet and bile salts, allowing for the selection and isolation of gram-negative",
   'predominantly made from the lactose sugar in the agar.\\n\\n\\n== Variant ==\\nA variant, s

In [62]:
result = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
)
print(result)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.8203}


In [63]:
result.to_pandas()

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,what bacteria grow on macconkey agar,[context: ['MacConkey agar is a selective and ...,MacConkey agar is designed to selectively isol...,1.0,0.838043
1,who wrote a rose is a rose is a rose,[she would carve on the tree Rose is a Rose is...,"Gertrude Stein wrote the sentence ""A rose is a...",1.0,0.802468
